In [ ]:
# Compare exact Parquet quantiles against multiple sketch approximations for a chosen typ (uid or gid).
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PARQUET_UID_DIR = "/home/ubuntu/dew/auxiliary/pyspark128-out2/hpss/hpss-uid-sorted-parquet"
PARQUET_GID_DIR = "/home/ubuntu/dew/auxiliary/pyspark128-out2/hpss/hpss-gid-sorted-parquet"
TYP = "gid"  # set to "uid" or "gid"
ALGOS = ["dd", "kll", "req", "td"]
RANK_ERRORS_OUT_TEMPLATE = "rank_errors_{typ}.{algo}.parquet"
EXACT_GROUPS_OUT_TEMPLATE = "exact_groups_{typ}.parquet"
QUANTILES = {
    "p10": 0.10,
    "p25": 0.25,
    "p50": 0.50,
    "p75": 0.75,
    "p90": 0.90,
    "p99": 0.99,
}


In [ ]:
def load_parquet_dataset(parquet_dir: str, typ: str, id_col: str) -> pd.DataFrame:
    """Load one parquet dataset into normalized DataFrame with typ/id/metric/value."""
    dataset = pq.ParquetDataset(parquet_dir)
    table = dataset.read()
    df = table.to_pandas()
    df = df.copy()
    df["typ"] = typ
    df["id"] = df[id_col].astype("int64")
    df["metric"] = df["metric"].astype(str)
    df["value"] = pd.to_numeric(df["value"], errors="coerce").astype("int64")
    return df[["typ", "id", "metric", "value"]]


In [ ]:
def sketches_csvs(abbv: str, typ: str): # dd, kll, req, td
    return [
        f"hpss-approx/{typ}-{abbv}-1.csv",
        f"hpss-approx/{typ}-{abbv}-2.csv",
        f"hpss-approx/{typ}-{abbv}-3.csv",
    ]


def load_all_sketches(paths, typ: str):
    rows = []
    id_col = "uid" if typ == "uid" else "gid"
    for path in paths:
        df = pd.read_csv(path)
        run = Path(path).stem
        df = df.rename(columns={id_col: "id"})
        df["id"] = df["id"].astype("int64")
        df["metric"] = df["metric"].astype(str)
        df["typ"] = typ
        df["run"] = run
        rows.append(
            df[["typ", "id", "metric", "run", "p10", "p25", "p50", "p75", "p90", "p99"]]
        )
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


In [ ]:
def prepare_exact_groups(exact_df: pd.DataFrame):
    """Group exact values by (typ, id, metric) and pre-sort values."""
    exact_df = exact_df.copy()
    exact_df["typ"] = exact_df["typ"].astype(str)
    exact_df["id"] = exact_df["id"].astype("int64")
    exact_df["metric"] = exact_df["metric"].astype(str)
    exact_df["value"] = pd.to_numeric(exact_df["value"], errors="coerce").astype(
        "int64"
    )

    groups = {}
    for key, values in exact_df.groupby(["typ", "id", "metric"])["value"]:
        vals = np.sort(values.to_numpy())
        groups[key] = {"values": vals, "N": len(vals)}
    return groups


In [ ]:
# Compute rank/value errors per (typ, id, metric, run) using pre-grouped exact inputs.
def compute_rank_errors_by_typ_run(
    exact_groups, approx_df: pd.DataFrame, quantiles=QUANTILES
) -> pd.DataFrame:
    if approx_df.empty:
        return pd.DataFrame()

    approx_df = approx_df.copy()
    approx_df["typ"] = approx_df["typ"].astype(str)
    approx_df["id"] = approx_df["id"].astype("int64")
    approx_df["metric"] = approx_df["metric"].astype(str)
    approx_df["run"] = approx_df["run"].astype(str)
    approx_indexed = approx_df.set_index(["typ", "id", "metric", "run"])

    results = []
    for (typ, id_val, metric), group in exact_groups.items():
        try:
            approx_rows = approx_indexed.loc[(typ, id_val, metric)]
        except KeyError:
            continue  # no approx rows for this key

        if isinstance(approx_rows, pd.Series):
            approx_rows = approx_rows.to_frame().T

        vals = group["values"]
        N = group["N"]
        for run, approx_row in approx_rows.iterrows():
            for col, p in quantiles.items():
                if col not in approx_row:
                    continue
                q_hat = int(approx_row[col])
                idx = int(p * (N - 1))
                q_exact = int(vals[idx])

                rank_hat = int(np.searchsorted(vals, q_hat, side="right"))
                expected_rank = idx + 1
                rank_error = abs(rank_hat - expected_rank)
                rank_error_norm = rank_error / N

                abs_val_err = abs(q_hat - q_exact)
                rel_value_error = abs_val_err / abs(q_exact) if q_exact != 0 else np.nan

                results.append(
                    {
                        "typ": typ,
                        "id": int(id_val),
                        "metric": metric,
                        "run": run,
                        "quantile": col,
                        "p": p,
                        "exact": q_exact,
                        "approx": q_hat,
                        "rank_hat": rank_hat,
                        "expected_rank": expected_rank,
                        "rank_error": rank_error,
                        "rank_error_norm": rank_error_norm,
                        "N": N,
                        "abs_val_err": abs_val_err,
                        "rel_value_error": rel_value_error,
                    }
                )
    return pd.DataFrame(results)


def summarize_run_errors(rank_errors: pd.DataFrame, group_keys=None, min_n: int | None = None) -> pd.DataFrame:
    if rank_errors.empty:
        return rank_errors
    if min_n is not None:
        rank_errors = rank_errors[rank_errors["N"] >= min_n]
        if rank_errors.empty:
            return rank_errors
    keys = group_keys if group_keys is not None else ["typ", "id", "metric", "quantile"]
    grouped = (
        rank_errors.groupby(keys)
        .agg(
            run_count=("run", "nunique"),
            rank_error_norm_mean=("rank_error_norm", "mean"),
            rank_error_norm_max=("rank_error_norm", "max"),
            rel_value_error_mean=("rel_value_error", "mean"),
            rel_value_error_max=("rel_value_error", "max"),
        )
        .reset_index()
    )
    return grouped


In [ ]:
# Load exacts for the selected typ and precompute sorted value groups.
parquet_dir = PARQUET_UID_DIR if TYP == "uid" else PARQUET_GID_DIR
id_col = "uid" if TYP == "uid" else "gid"

exact_df = load_parquet_dataset(parquet_dir, TYP, id_col)
print(f"Exact df ({TYP}): {exact_df.shape}")

exact_groups = prepare_exact_groups(exact_df)
print(f"Prepared exact groups: {len(exact_groups)}")


In [ ]:
# Run rank error calculations for each algorithm and write parquet outputs.
rank_errors_outputs = []
for algo in ALGOS:
    print(f"\n=== {algo} ({TYP}) ===")
    approx_df = load_all_sketches(sketches_csvs(algo, TYP), TYP)
    print(f"Approx df ({algo}, {TYP}): {approx_df.shape}")

    rank_errors = compute_rank_errors_by_typ_run(exact_groups, approx_df)
    print(f"Rank errors ({algo}, {TYP}): {rank_errors.shape}")

    out_path = RANK_ERRORS_OUT_TEMPLATE.format(typ=TYP, algo=algo)
    rank_errors.to_parquet(out_path, index=False)
    print(f"Wrote {out_path}")

    rank_errors_outputs.append(rank_errors.assign(algo=algo))


In [ ]:
# Load saved rank error parquet outputs for uid+gid per algo, then summarize by quantile.
from IPython.display import display

summaries = {}
for algo in ALGOS:
    uid_path = RANK_ERRORS_OUT_TEMPLATE.format(typ="uid", algo=algo)
    gid_path = RANK_ERRORS_OUT_TEMPLATE.format(typ="gid", algo=algo)
    uid_df = pd.read_parquet(uid_path)
    gid_df = pd.read_parquet(gid_path)
    print(f"Loaded {uid_path}: {uid_df.shape}")
    print(f"Loaded {gid_path}: {gid_df.shape}")

    rank_errors = pd.concat([uid_df, gid_df], ignore_index=True)
    print(f"Combined ({algo}) rank_errors: {rank_errors.shape}")
    rank_errors["algo"] = algo

    grouped = summarize_run_errors(rank_errors, group_keys=["quantile"], min_n=100)
    pd.set_option("display.float_format", lambda x: f"{x:.4f}")
    print(f"Grouped rank errors for {algo}:")
    display(grouped)
    summaries[algo] = grouped
